# 05_final_data_prepare.ipynb

Build the **final router training datasets** (train / val / test) using **EDA-informed best practices**:

## Key Improvements from EDA:

1. **Hierarchical Performance Scoring** (from 02_perf_function_change.ipynb):
   - Combines sample-level score + task-level priors + global priors
   - Achieves ~10-15% better accuracy at same cost vs. linear scoring
   - Weights: w_sample=0.7, w_task=0.2, w_global=0.1

2. **Utility-Based Label Selection** (from 03_cost_analysis.ipynb):
   - Linear utility: `utility = perf - λ * cost` with λ=10000
   - Balances performance and cost optimally
   - Produces meaningful separation between models

3. **Complete Feature Set** (from your specification):
   - Vision: image_path (for encoder input)
   - Text: prompt_raw, router_task, source_dataset, txt_question_type
   - Metadata: image dimensions, aspect ratio, prompt length
   - Labels: hard labels (best_model_id) + soft labels (probability distribution)

## Output:
- `router_train_final.parquet`
- `router_val_final.parquet`
- `router_test_final.parquet`

Each row contains all features needed for multimodal router training.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from imports.common_utils import return_model_specs, return_model_pricing
from imports.performance_utils import (
    PerfWeightsHier,
    PriorConfig,
    add_sample_scores_for_models,
    add_valid_mask_columns,
    build_long_perf_dataframe,
    build_perf_matrix_hierarchical,
    build_valid_matrix,
    compute_global_prior,
    compute_task_prior,
)
from imports.cost_utils import (
    add_cost_columns_for_models,
    build_cost_matrix,
    compute_utility_matrix,
)

# Import image fetching utility
import sys
sys.path.insert(0, str(Path.cwd().parent.parent))
from imports.check_data_utils import fetch_cauldron_image, DEFAULT_IMAGE_ROOT

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1. Configuration

In [3]:
# Paths
DATA_ROOT = Path.cwd().parent.parent.parent / "dataset" / "final_dataset"
TRAIN_PIVOT_PATH = DATA_ROOT / 'router_pivot_dataset_train.parquet'
VAL_PIVOT_PATH   = DATA_ROOT / 'router_pivot_dataset_validation.parquet'
TEST_PIVOT_PATH  = DATA_ROOT / 'router_pivot_dataset_test.parquet'

OUT_DIR = DATA_ROOT / 'router_final'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Image cache directory (for fetching and saving images from Cauldron)
IMAGE_ROOT = Path.cwd().parent.parent.parent / "dataset" / "which_vlm_data" / "images" / "cauldron"
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Data root: {DATA_ROOT}')
print(f'Output directory: {OUT_DIR}')
print(f'Image cache directory: {IMAGE_ROOT}')

Data root: /storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/final_dataset
Output directory: /storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/final_dataset/router_final
Image cache directory: /storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/which_vlm_data/images/cauldron


In [4]:
# Model configuration
MODEL_SPECS = return_model_specs()
MODEL_PRICING = return_model_pricing()
model_names = [m['name'] for m in MODEL_SPECS]
ID_TO_NAME = {m['id']: m['name'] for m in MODEL_SPECS}
NAME_TO_ID = {m['name']: m['id'] for m in MODEL_SPECS}

Configured models:
  id=0 name=deepseek_ocr prefix=deepseek_ocr__
  id=1 name=qwen2_5_vl_3b prefix=qwen2_5_vl_3b__
  id=2 name=qwen2_5_vl_7b prefix=qwen2_5_vl_7b__
  id=3 name=qwen3_vl_8b_thinking prefix=qwen3_vl_8b_thinking__
  id=4 name=gemma_3_27b prefix=gemma_3_27b__
Configured model pricing (USD per 1K tokens):
  deepseek_ocr: prompt=$3e-05, completion=$0.0001
  qwen2_5_vl_3b: prompt=$0.0001, completion=$0.0001
  qwen2_5_vl_7b: prompt=$0.0002, completion=$0.0002
  qwen3_vl_8b_thinking: prompt=$0.00018, completion=$0.0021
  gemma_3_27b: prompt=$9e-05, completion=$0.00016


In [5]:
# Hierarchical performance weights (from EDA)
# These weights were validated in 02_perf_function_change.ipynb
HIER_WEIGHTS = PerfWeightsHier(
    w_sample=0.7,  # Sample-level performance dominates
    w_task=0.2,    # Task-level priors help with task-specific patterns
    w_global=0.1,  # Global priors provide weak baseline
)

# Prior configuration
PRIOR_CONFIG = PriorConfig(
    router_task_col='router_task',
    model_col='model_name',
    perf_col='sample_score',
    alpha=50.0,  # Smoothing strength
)

In [6]:
# Utility configuration (from 03_cost_analysis.ipynb)
# Linear utility with λ=10000 provides best cost-performance balance
UTILITY_SCHEME = 'linear'
LAMBDA_COST = 10000.0

# Soft labels configuration
USE_SOFT_LABELS = True
SOFTMAX_TEMPERATURE = 0.3  # Sharper distribution emphasizes best models

print(f'Utility scheme: {UTILITY_SCHEME} with λ={LAMBDA_COST}')
print(f'Soft labels: {"enabled" if USE_SOFT_LABELS else "disabled"} (T={SOFTMAX_TEMPERATURE})')

Utility scheme: linear with λ=10000.0
Soft labels: enabled (T=0.3)


## 2. Load Data

In [7]:
print('Loading pivot datasets...')
train_pivot = pd.read_parquet(TRAIN_PIVOT_PATH)
val_pivot   = pd.read_parquet(VAL_PIVOT_PATH)
test_pivot  = pd.read_parquet(TEST_PIVOT_PATH)

print(f'Train pivot rows: {len(train_pivot):,}')
print(f'Val pivot rows:   {len(val_pivot):,}')
print(f'Test pivot rows:  {len(test_pivot):,}')

Loading pivot datasets...
Train pivot rows: 63,963
Val pivot rows:   13,706
Test pivot rows:  13,707


## 3. Compute Priors from Training Data

We compute task-level and global priors **only from training data** to avoid data leakage.

In [8]:
# Add sample scores to training data
print('Computing sample scores for training data...')
train_with_scores = add_valid_mask_columns(train_pivot, MODEL_SPECS)
train_with_scores = add_sample_scores_for_models(train_with_scores, model_names)

# Convert to long format for prior computation
train_long = build_long_perf_dataframe(
    train_with_scores,
    model_names=model_names,
    router_task_col=PRIOR_CONFIG.router_task_col,
    value_suffix='__sample_score',
)

print(f'Training long format: {len(train_long):,} rows')
print(f'Unique tasks: {train_long["router_task"].nunique()}')
print(f'Unique models: {train_long["model_name"].nunique()}')

Computing sample scores for training data...
Training long format: 319,815 rows
Unique tasks: 30
Unique models: 5


In [9]:
# Compute global and task priors from training data only
print('\nComputing priors from training data...')
global_prior_df = compute_global_prior(train_long, PRIOR_CONFIG)
task_prior_df = compute_task_prior(train_long, global_prior_df, PRIOR_CONFIG)

print('\nGlobal priors per model:')
display(global_prior_df)

print('\nTask priors (first 10 rows):')
display(task_prior_df.head(10))


Computing priors from training data...

Global priors per model:


,model_name,global_prior
0,deepseek_ocr,0.287532
1,gemma_3_27b,0.531274
2,qwen2_5_vl_3b,0.773608
3,qwen2_5_vl_7b,0.768850
4,qwen3_vl_8b_thinking,0.576505



Task priors (first 10 rows):


,router_task,model_name,task_prior,n_task
0,abstract_reasoning,deepseek_ocr,0.425090,1401
1,abstract_reasoning,gemma_3_27b,0.648638,1401
2,abstract_reasoning,qwen2_5_vl_3b,0.759635,1401
3,abstract_reasoning,qwen2_5_vl_7b,0.677253,1401
4,abstract_reasoning,qwen3_vl_8b_thinking,0.593902,1401
5,chart_captioning,deepseek_ocr,0.107984,2795
6,chart_captioning,gemma_3_27b,0.189817,2795
7,chart_captioning,qwen2_5_vl_3b,0.729495,2795
8,chart_captioning,qwen2_5_vl_7b,0.302249,2795
9,chart_captioning,qwen3_vl_8b_thinking,0.173930,2795


## 4. Core Processing Function

This function:
1. Computes hierarchical performance scores
2. Computes true costs per model
3. Applies utility-based routing to select best models
4. Generates hard and soft labels
5. Extracts all features needed for router training

In [10]:
def process_split(
    df_raw: pd.DataFrame,
    split_name: str,
    global_prior_df: pd.DataFrame,
    task_prior_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Process one dataset split into router-ready format.
    
    Steps:
    1. Add valid masks and sample scores
    2. Build hierarchical performance matrix
    3. Build cost matrix
    4. Compute utility and select best models
    5. Extract input features and labels
    6. Fetch and cache images from Cauldron
    """
    print(f'\n=== Processing split: {split_name} ===')
    print(f'Input rows: {len(df_raw):,}')
    
    # Step 1: Add valid masks and sample scores
    df_proc = add_valid_mask_columns(df_raw, MODEL_SPECS)
    df_proc = add_sample_scores_for_models(df_proc, model_names)
    
    # Step 2: Build hierarchical performance matrix [N, M]
    perf_mat = build_perf_matrix_hierarchical(
        df_proc=df_proc,
        model_names=model_names,
        router_task_col=PRIOR_CONFIG.router_task_col,
        task_prior_df=task_prior_df,
        global_prior_df=global_prior_df,
        prior_cfg=PRIOR_CONFIG,
        weights=HIER_WEIGHTS,
    )
    print(f'Performance matrix shape: {perf_mat.shape}')
    
    # Step 3: Build cost matrix [N, M]
    df_proc = add_cost_columns_for_models(df_proc, MODEL_SPECS, MODEL_PRICING)
    cost_mat = build_cost_matrix(df_proc, model_names)
    print(f'Cost matrix shape: {cost_mat.shape}')
    
    # Step 4: Build valid mask and compute utility
    valid_mat = build_valid_matrix(df_proc, model_names)
    util_mat = compute_utility_matrix(
        perf_mat,
        cost_mat,
        scheme=UTILITY_SCHEME,
        lambda_cost=LAMBDA_COST,
    )
    
    # Mask invalid entries
    util_mat = np.where(valid_mat, util_mat, -1e12)
    
    # Step 5: Select best models (hard labels)
    best_idx = util_mat.argmax(axis=1)  # [N]
    idx_arr = np.arange(len(df_proc))
    
    df_proc['router_best_model_id'] = best_idx
    df_proc['router_best_model_name'] = pd.Series(best_idx).map(ID_TO_NAME)
    
    # Store performance and cost of chosen model
    df_proc['router_chosen_perf'] = perf_mat[idx_arr, best_idx]
    df_proc['router_chosen_cost'] = cost_mat[idx_arr, best_idx]
    
    # Check if any model is valid
    df_proc['router_valid_any_model'] = valid_mat.any(axis=1)
    
    # Step 6: Generate soft labels (optional)
    if USE_SOFT_LABELS:
        # For each sample, compute softmax over utilities of valid models
        soft_probs = np.zeros((len(df_proc), len(model_names)), dtype=float)
        
        for i in range(len(df_proc)):
            valid_mask_i = valid_mat[i]
            if valid_mask_i.any():
                util_i = util_mat[i].copy()
                util_i[~valid_mask_i] = -1e12
                
                # Softmax with temperature
                logits = util_i / SOFTMAX_TEMPERATURE
                logits = logits - logits.max()  # Numerical stability
                exp_logits = np.exp(logits)
                exp_logits[~valid_mask_i] = 0.0
                probs = exp_logits / (exp_logits.sum() + 1e-12)
                soft_probs[i] = probs
        
        # Add soft label columns
        for j, name in enumerate(model_names):
            df_proc[f'router_soft_p_{name}'] = soft_probs[:, j]
    
    # Step 7: Filter invalid samples
    before = len(df_proc)
    df_proc = df_proc[df_proc['router_valid_any_model']].reset_index(drop=True)
    after = len(df_proc)
    print(f'Filtered invalid rows: {before - after} dropped; {after:,} remain.')
    
    # Step 8: Fetch and cache images from Cauldron
    df_proc = fetch_and_cache_images(df_proc, IMAGE_ROOT, max_workers=32)
    
    # Step 9: Extract router input features
    input_cols = [
        'sample_id',
        'image_path',
        'prompt_raw',
        'router_task',
        'source_dataset',
        'source_config',
        'txt_question_type',
        'txt_has_mc_options',
        'img_width',
        'img_height',
        'img_aspect_ratio',
        'txt_prompt_length_chars',
        'txt_prompt_length_words',
        'ground_truth',
        'ground_truth_type',
    ]
    input_cols = [c for c in input_cols if c in df_proc.columns]
    
    # Label columns
    label_cols = [
        'router_best_model_id',
        'router_best_model_name',
        'router_chosen_perf',
        'router_chosen_cost',
    ]
    
    # Soft label columns
    soft_label_cols = []
    if USE_SOFT_LABELS:
        soft_label_cols = [f'router_soft_p_{name}' for name in model_names]
    
    # Diagnostic columns (performance/cost per model)
    diagnostic_cols = []
    for name in model_names:
        for suffix in ['__sample_score', '__cost', '__valid_mask', '__is_correct', '__score_f1']:
            col = f'{name}{suffix}'
            if col in df_proc.columns:
                diagnostic_cols.append(col)
    
    # Add metadata columns for image fetching
    metadata_cols = ['image_bytes_hash', 'source_index']
    metadata_cols = [c for c in metadata_cols if c in df_proc.columns]
    
    # Combine all columns
    all_cols = input_cols + label_cols + soft_label_cols + diagnostic_cols + metadata_cols
    router_df = df_proc[all_cols].copy()
    
    print(f'Final router_df shape: {router_df.shape}')
    print(f'  - Input features: {len(input_cols)}')
    print(f'  - Label columns: {len(label_cols)}')
    print(f'  - Soft labels: {len(soft_label_cols)}')
    print(f'  - Diagnostic columns: {len(diagnostic_cols)}')
    print(f'  - Metadata columns: {len(metadata_cols)}')
    
    # Step 10: Save
    out_path = OUT_DIR / f'router_{split_name}_final.parquet'
    router_df.to_parquet(out_path, index=False)
    print(f'Saved to: {out_path}')
    
    return router_df

In [11]:
def process_split(
    df_raw: pd.DataFrame,
    split_name: str,
    global_prior_df: pd.DataFrame,
    task_prior_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Process one dataset split into router-ready format.
    
    Steps:
    1. Add valid masks and sample scores
    2. Build hierarchical performance matrix
    3. Build cost matrix
    4. Compute utility and select best models
    5. Extract input features and labels
    """
    print(f'\n=== Processing split: {split_name} ===')
    print(f'Input rows: {len(df_raw):,}')
    
    # Step 1: Add valid masks and sample scores
    df_proc = add_valid_mask_columns(df_raw, MODEL_SPECS)
    df_proc = add_sample_scores_for_models(df_proc, model_names)
    
    # Step 2: Build hierarchical performance matrix [N, M]
    perf_mat = build_perf_matrix_hierarchical(
        df_proc=df_proc,
        model_names=model_names,
        router_task_col=PRIOR_CONFIG.router_task_col,
        task_prior_df=task_prior_df,
        global_prior_df=global_prior_df,
        prior_cfg=PRIOR_CONFIG,
        weights=HIER_WEIGHTS,
    )
    print(f'Performance matrix shape: {perf_mat.shape}')
    
    # Step 3: Build cost matrix [N, M]
    df_proc = add_cost_columns_for_models(df_proc, MODEL_SPECS, MODEL_PRICING)
    cost_mat = build_cost_matrix(df_proc, model_names)
    print(f'Cost matrix shape: {cost_mat.shape}')
    
    # Step 4: Build valid mask and compute utility
    valid_mat = build_valid_matrix(df_proc, model_names)
    util_mat = compute_utility_matrix(
        perf_mat,
        cost_mat,
        scheme=UTILITY_SCHEME,
        lambda_cost=LAMBDA_COST,
    )
    
    # Mask invalid entries
    util_mat = np.where(valid_mat, util_mat, -1e12)
    
    # Step 5: Select best models (hard labels)
    best_idx = util_mat.argmax(axis=1)  # [N]
    idx_arr = np.arange(len(df_proc))
    
    df_proc['router_best_model_id'] = best_idx
    df_proc['router_best_model_name'] = pd.Series(best_idx).map(ID_TO_NAME)
    
    # Store performance and cost of chosen model
    df_proc['router_chosen_perf'] = perf_mat[idx_arr, best_idx]
    df_proc['router_chosen_cost'] = cost_mat[idx_arr, best_idx]
    
    # Check if any model is valid
    df_proc['router_valid_any_model'] = valid_mat.any(axis=1)
    
    # Step 6: Generate soft labels (optional)
    if USE_SOFT_LABELS:
        # For each sample, compute softmax over utilities of valid models
        soft_probs = np.zeros((len(df_proc), len(model_names)), dtype=float)
        
        for i in range(len(df_proc)):
            valid_mask_i = valid_mat[i]
            if valid_mask_i.any():
                util_i = util_mat[i].copy()
                util_i[~valid_mask_i] = -1e12
                
                # Softmax with temperature
                logits = util_i / SOFTMAX_TEMPERATURE
                logits = logits - logits.max()  # Numerical stability
                exp_logits = np.exp(logits)
                exp_logits[~valid_mask_i] = 0.0
                probs = exp_logits / (exp_logits.sum() + 1e-12)
                soft_probs[i] = probs
        
        # Add soft label columns
        for j, name in enumerate(model_names):
            df_proc[f'router_soft_p_{name}'] = soft_probs[:, j]
    
    # Step 7: Filter invalid samples
    before = len(df_proc)
    df_proc = df_proc[df_proc['router_valid_any_model']].reset_index(drop=True)
    after = len(df_proc)
    print(f'Filtered invalid rows: {before - after} dropped; {after:,} remain.')
    
    # Step 8: Extract router input features
    input_cols = [
        'sample_id',
        'image_path',
        'prompt_raw',
        'router_task',
        'source_dataset',
        'source_config',
        'txt_question_type',
        'txt_has_mc_options',
        'img_width',
        'img_height',
        'img_aspect_ratio',
        'txt_prompt_length_chars',
        'txt_prompt_length_words',
        'ground_truth',
        'ground_truth_type',
    ]
    input_cols = [c for c in input_cols if c in df_proc.columns]
    
    # Label columns
    label_cols = [
        'router_best_model_id',
        'router_best_model_name',
        'router_chosen_perf',
        'router_chosen_cost',
    ]
    
    # Soft label columns
    soft_label_cols = []
    if USE_SOFT_LABELS:
        soft_label_cols = [f'router_soft_p_{name}' for name in model_names]
    
    # Diagnostic columns (performance/cost per model)
    diagnostic_cols = []
    for name in model_names:
        for suffix in ['__sample_score', '__cost', '__valid_mask', '__is_correct', '__score_f1']:
            col = f'{name}{suffix}'
            if col in df_proc.columns:
                diagnostic_cols.append(col)
    
    # Combine all columns
    all_cols = input_cols + label_cols + soft_label_cols + diagnostic_cols
    router_df = df_proc[all_cols].copy()
    
    print(f'Final router_df shape: {router_df.shape}')
    print(f'  - Input features: {len(input_cols)}')
    print(f'  - Label columns: {len(label_cols)}')
    print(f'  - Soft labels: {len(soft_label_cols)}')
    print(f'  - Diagnostic columns: {len(diagnostic_cols)}')
    
    # Step 9: Save
    out_path = OUT_DIR / f'router_{split_name}_final.parquet'
    router_df.to_parquet(out_path, index=False)
    print(f'Saved to: {out_path}')
    
    return router_df

## 5. Process All Splits

In [12]:
router_train = process_split(train_pivot, 'train', global_prior_df, task_prior_df)
router_val   = process_split(val_pivot, 'val', global_prior_df, task_prior_df)
router_test  = process_split(test_pivot, 'test', global_prior_df, task_prior_df)


=== Processing split: train ===
Input rows: 63,963
Performance matrix shape: (63963, 5)
Cost matrix shape: (63963, 5)
Filtered invalid rows: 0 dropped; 63,963 remain.
Final router_df shape: (63963, 49)
  - Input features: 15
  - Label columns: 4
  - Soft labels: 5
  - Diagnostic columns: 25
Saved to: /storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/final_dataset/router_final/router_train_final.parquet

=== Processing split: val ===
Input rows: 13,706
Performance matrix shape: (13706, 5)
Cost matrix shape: (13706, 5)
Filtered invalid rows: 0 dropped; 13,706 remain.
Final router_df shape: (13706, 49)
  - Input features: 15
  - Label columns: 4
  - Soft labels: 5
  - Diagnostic columns: 25
Saved to: /storage/ice1/1/0/vchopra37/projects/vlm_router/dataset/final_dataset/router_final/router_val_final.parquet

=== Processing split: test ===
Input rows: 13,707
Performance matrix shape: (13707, 5)
Cost matrix shape: (13707, 5)
Filtered invalid rows: 0 dropped; 13,707 remain.
Final route

## 6. Inspect Results

In [13]:
print('\n=== Summary Statistics ===')
print(f'\nTrain set: {len(router_train):,} samples')
print(f'Val set:   {len(router_val):,} samples')
print(f'Test set:  {len(router_test):,} samples')

print('\n=== Model Selection Distribution (Training) ===')
print(router_train['router_best_model_name'].value_counts())
print('\nFractions:')
print(router_train['router_best_model_name'].value_counts(normalize=True))


=== Summary Statistics ===

Train set: 63,963 samples
Val set:   13,706 samples
Test set:  13,707 samples

=== Model Selection Distribution (Training) ===
router_best_model_name
qwen2_5_vl_3b    27795
deepseek_ocr     17921
gemma_3_27b      16723
qwen2_5_vl_7b     1524
Name: count, dtype: int64

Fractions:
router_best_model_name
qwen2_5_vl_3b    0.434548
deepseek_ocr     0.280178
gemma_3_27b      0.261448
qwen2_5_vl_7b    0.023826
Name: proportion, dtype: float64


In [14]:
print('\n=== Performance & Cost Statistics (Training) ===')
print(f"Mean chosen performance: {router_train['router_chosen_perf'].mean():.4f}")
print(f"Mean chosen cost: {router_train['router_chosen_cost'].mean():.6f} USD")

print('\n=== Per-model statistics ===')
for name in model_names:
    mask = router_train['router_best_model_name'] == name
    if mask.sum() > 0:
        avg_perf = router_train.loc[mask, 'router_chosen_perf'].mean()
        avg_cost = router_train.loc[mask, 'router_chosen_cost'].mean()
        count = mask.sum()
        print(f'{name:25s}: {count:6,} samples | perf={avg_perf:.4f} | cost=${avg_cost:.6f}')


=== Performance & Cost Statistics (Training) ===
Mean chosen performance: 0.7511
Mean chosen cost: 0.000037 USD

=== Per-model statistics ===
deepseek_ocr             : 17,921 samples | perf=0.5412 | cost=$0.000025
qwen2_5_vl_3b            : 27,795 samples | perf=0.8738 | cost=$0.000041
qwen2_5_vl_7b            :  1,524 samples | perf=0.9395 | cost=$0.000057
gemma_3_27b              : 16,723 samples | perf=0.7550 | cost=$0.000040


In [15]:
print('\n=== Example Training Row ===')
example = router_train.sample(1, random_state=42)

# Show input features
print('\nInput Features:')
input_features = [
    'sample_id', 'image_path', 'prompt_raw', 'router_task',
    'txt_question_type', 'img_width', 'img_height', 'img_aspect_ratio',
    'txt_prompt_length_chars', 'txt_prompt_length_words'
]
for col in input_features:
    if col in example.columns:
        print(f'  {col:30s}: {example[col].values[0]}')

# Show labels
print('\nLabels:')
print(f'  router_best_model_id:   {example["router_best_model_id"].values[0]}')
print(f'  router_best_model_name: {example["router_best_model_name"].values[0]}')
print(f'  router_chosen_perf:     {example["router_chosen_perf"].values[0]:.4f}')
print(f'  router_chosen_cost:     ${example["router_chosen_cost"].values[0]:.6f}')

# Show soft labels
if USE_SOFT_LABELS:
    print('\nSoft Labels (probability distribution):')
    for name in model_names:
        col = f'router_soft_p_{name}'
        if col in example.columns:
            prob = example[col].values[0]
            print(f'  {name:25s}: {prob:.4f}')

# Show the image for this sample
print('\nImage preview:')
from IPython.display import display
from PIL import Image

def _display(path: Path):
    img = Image.open(path)
    img.load()
    display(img)

image_path_val = example.get('image_path', [None])[0] if 'image_path' in example.columns else None
if image_path_val and Path(image_path_val).exists():
    path = Path(image_path_val)
    _display(path)
    print(f'  Image source: {path}')
else:
    sample_id = example['sample_id'].values[0]
    raw_row = train_pivot.loc[train_pivot['sample_id'] == sample_id]
    if raw_row.empty:
        print('  Could not find source row in pivot dataset.')
    else:
        raw_row = raw_row.iloc[0]
        image_hash = raw_row.get('image_bytes_hash')
        source_config = raw_row.get('source_config')
        source_index = raw_row.get('source_index')
        cache_root_str = raw_row.get('image_cache_root')
        cache_root = Path(cache_root_str) if pd.notna(cache_root_str) and cache_root_str else IMAGE_ROOT
        if not cache_root.exists():
            cache_root = IMAGE_ROOT

        cached_path = None
        if pd.notna(image_hash) and pd.notna(source_config):
            candidate = cache_root / str(source_config) / f'{image_hash}.png'
            if candidate.exists():
                cached_path = candidate

        try:
            if cached_path:
                _display(cached_path)
                print(f'  Image source: {cached_path}')
            elif pd.notna(source_config) and pd.notna(source_index):
                img, _ = fetch_cauldron_image(
                    str(source_config),
                    int(source_index),
                    image_hash=image_hash if pd.notna(image_hash) else None,
                    prefer_local_cache=True,
                    image_root=cache_root,
                )
                display(img)
                filename = f"{image_hash}.png" if pd.notna(image_hash) else f"{int(source_index):08d}.png"
                print(f'  Image fetched and cached at {cache_root / str(source_config) / filename}')
            else:
                print('  No source_config/source_index available to fetch image.')
        except Exception as e:
            print(f'  [image display failed] {e}')



=== Example Training Row ===

Input Features:
  sample_id                     : visual7w_01175_cbd2a043b31fbcbd
  image_path                    : None
  prompt_raw                    : Question: who would drive this vehicle?
Choices:
A. A trucker.
B. A farmer.
C. A bus driver.
D. A plumber.
Answer with the letter.
  router_task                   : general_vqa
  txt_question_type             : None
  img_width                     : 500.0
  img_height                    : 375.0
  img_aspect_ratio              : 1.333
  txt_prompt_length_chars       : 130.0
  txt_prompt_length_words       : 24.0

Labels:
  router_best_model_id:   1
  router_best_model_name: qwen2_5_vl_3b
  router_chosen_perf:     0.4682
  router_chosen_cost:     $0.000029

Soft Labels (probability distribution):
  deepseek_ocr             : 0.0000
  qwen2_5_vl_3b            : 0.6067
  qwen2_5_vl_7b            : 0.2342
  qwen3_vl_8b_thinking     : 0.0003
  gemma_3_27b              : 0.1587


## 7. Validation: Routing Distribution by Task

In [18]:
import plotly.express as px

# Routing distribution per task
task_routing = (
    router_train
    .groupby('router_task')['router_best_model_name']
    .value_counts(normalize=True)
    .rename('fraction')
    .reset_index()
)

fig = px.bar(
    task_routing,
    x='router_task',
    y='fraction',
    color='router_best_model_name',
    title='Model routing distribution per task (training set)',
    labels={'fraction': 'Fraction of samples'},
    barmode='stack',
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

## 8. Summary

### What we built:

1. **Hierarchical Performance Scoring**: Combines sample, task, and global signals
2. **Utility-Based Selection**: Balances performance and cost using linear utility
3. **Complete Feature Set**: All inputs needed for multimodal router training
4. **Hard + Soft Labels**: Enables both cross-entropy and KL-divergence training
5. **Image Fetching & Caching**: Fetches images from Cauldron and caches them locally

### How to use this dataset:

During router training:
1. **Inputs** (features):
   - `image_path` → Load actual image from local cache (fetched from Cauldron)
   - `prompt_raw` → text encoder
   - `router_task`, `txt_question_type` → categorical embeddings
   - Image metadata: `img_width`, `img_height`, `img_aspect_ratio`
   - Text metadata: `txt_prompt_length_chars`, `txt_prompt_length_words`

2. **Labels**:
   - Hard: `router_best_model_id` (classification target)
   - Soft: `router_soft_p_*` columns (KL divergence target)

3. **Training loss**:
   - Option A: Pure CE loss on hard labels
   - Option B: CE loss + KL divergence to soft labels
   - Option C: Pure KL divergence (label smoothing effect)

### Image Fetching Process:

The dataset preparation now includes automatic image fetching and caching:

1. **Fetch from Cauldron**: For each sample, fetch the image using `source_config` + `source_index`
2. **Verify Hash**: Verify the image matches the expected `image_bytes_hash`
3. **Cache Locally**: Save to `{IMAGE_ROOT}/{source_config}/{image_hash}.png`
4. **Populate Path**: Store the local path in the `image_path` column
5. **Parallel Processing**: Uses 32 workers for efficient parallel fetching

**Image Cache Location:**
```
dataset/which_vlm_data/images/cauldron/
├── docvqa/
│   ├── {hash1}.png
│   ├── {hash2}.png
│   └── ...
├── vqav2/
│   └── ...
└── {other_configs}/
```

### Key improvements over original version:

1. Uses **hierarchical performance** (10-15% better than linear)
2. Uses **linear utility** with tuned λ (better cost-perf balance than lexico)
3. Includes **complete feature set** from EDA analysis
4. Generates **soft labels** for smoother training
5. Properly computed priors **from training data only** (no leakage)
6. **Fetches and caches images** from Cauldron automatically
7. **Populates image_path** for direct image loading during training

## 8. Summary

### What we built:

1. **Hierarchical Performance Scoring**: Combines sample, task, and global signals
2. **Utility-Based Selection**: Balances performance and cost using linear utility
3. **Complete Feature Set**: All inputs needed for multimodal router training
4. **Hard + Soft Labels**: Enables both cross-entropy and KL-divergence training

### How to use this dataset:

During router training:
1. **Inputs** (features):
   - `image_path` → vision encoder
   - `prompt_raw` → text encoder
   - `router_task`, `txt_question_type` → categorical embeddings
   - Image metadata: `img_width`, `img_height`, `img_aspect_ratio`
   - Text metadata: `txt_prompt_length_chars`, `txt_prompt_length_words`

2. **Labels**:
   - Hard: `router_best_model_id` (classification target)
   - Soft: `router_soft_p_*` columns (KL divergence target)

3. **Training loss**:
   - Option A: Pure CE loss on hard labels
   - Option B: CE loss + KL divergence to soft labels
   - Option C: Pure KL divergence (label smoothing effect)

### Key improvements over original version:

1. Uses **hierarchical performance** (10-15% better than linear)
2. Uses **linear utility** with tuned λ (better cost-perf balance than lexico)
3. Includes **complete feature set** from EDA analysis
4. Generates **soft labels** for smoother training
5. Properly computed priors **from training data only** (no leakage)